# Module 04 — Attention (notebook)

Walkthrough of [`attention.py`](attention.py). We'll:

1. Verify the reference scaled-dot-product matches PyTorch's `F.scaled_dot_product_attention`.
2. Build MHA, GQA, and MLA, run forward passes, check shapes.
3. Compare KV-cache size across variants for a Llama-3-70B-style and a DeepSeek-V3-style config.
4. (GPU only) Time FlashAttention against the naive implementation at a meaningful sequence length.

**Compute:** CPU is enough for sections 1–3. Section 4 needs a GPU for the timing to be informative.

**Time:** ~10 minutes.

In [ ]:
import math
import torch
import torch.nn.functional as F

from attention import (
    MultiHeadAttention,
    GroupedQueryAttention,
    MultiHeadLatentAttention,
    scaled_dot_product_reference,
)
from rope import precompute_freqs_cis

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

## 1. The reference matches SDPA

Verify that our pen-and-paper formula and PyTorch's optimized kernel agree numerically. (They should — FlashAttention is bit-equivalent to standard attention up to floating-point order-of-operations differences.)

In [ ]:
B, H, T, d = 2, 4, 16, 32
q = torch.randn(B, H, T, d)
k = torch.randn(B, H, T, d)
v = torch.randn(B, H, T, d)

out_ref = scaled_dot_product_reference(q, k, v, is_causal=True)
out_sdpa = F.scaled_dot_product_attention(q, k, v, is_causal=True)

max_diff = (out_ref - out_sdpa).abs().max().item()
print(f"max absolute difference: {max_diff:.2e}")
print(f"output shape:            {tuple(out_sdpa.shape)}")

Difference is on the order of `1e-7` — pure floating-point reordering. Use SDPA in real code; the reference is for understanding.

## 2. Build each variant, check forward-pass shapes

In [ ]:
B, T, d_model = 2, 64, 256
n_heads = 8
d_head = d_model // n_heads
x = torch.randn(B, T, d_model)
freqs_cis = precompute_freqs_cis(d_head, T)

mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
gqa = GroupedQueryAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=2)
mla = MultiHeadLatentAttention(
    d_model=d_model, n_heads=n_heads, d_head=d_head, d_rope=16, d_kv_latent=64,
)
freqs_cis_mla = precompute_freqs_cis(16, T)  # MLA uses d_rope=16, not d_head

out_mha = mha(x, freqs_cis)
out_gqa = gqa(x, freqs_cis)
out_mla = mla(x, freqs_cis_mla)

for name, mod, out in [("MHA", mha, out_mha), ("GQA(2)", gqa, out_gqa), ("MLA", mla, out_mla)]:
    n_params = sum(p.numel() for p in mod.parameters())
    print(f"{name:7s} output shape: {tuple(out.shape)}   params: {n_params:>8,}")

All three produce the same `(B, T, d_model)` output — they're drop-in interchangeable in a transformer block.

GQA has fewer parameters than MHA because K and V are smaller. MLA has roughly the same parameter count as GQA but a much smaller *cache*, which is the point.

## 3. KV-cache size at frontier scale

Per-token cache size is the actual cost driver. Let's compute it for two reference architectures.

In [ ]:
def kv_cache_summary(name, n_layers, per_token_bytes, context_len, batch_size=1):
    total_bytes = n_layers * per_token_bytes * context_len * batch_size
    return (
        f"{name:25s}  {per_token_bytes:>6,d} B/tok  "
        f"× {n_layers:>3d} layers × {context_len:>7,d} ctx  "
        f"= {total_bytes / 1e9:>7.2f} GB"
    )

# -- Llama-3-70B-style configuration --
# n_heads=64, n_kv_heads=8, d_head=128, n_layers=80, BF16
print("Llama-3-70B-style (n_heads=64, n_kv_heads=8, d_head=128, n_layers=80) at 64k context:\n")
print(kv_cache_summary("MHA (n_kv=n_q=64)",     80, 2 * 64 * 128 * 2, 65_536))
print(kv_cache_summary("GQA (n_kv=8)",          80, 2 *  8 * 128 * 2, 65_536))
print(kv_cache_summary("MQA (n_kv=1)",          80, 2 *  1 * 128 * 2, 65_536))

# -- DeepSeek-V3-style configuration --
# n_heads=128, d_head=128, d_kv_latent=512, d_rope=64, n_layers=61, BF16
print("\nDeepSeek-V3-style (n_heads=128, d_head=128, d_kv_latent=512, d_rope=64, n_layers=61) at 64k context:\n")
print(kv_cache_summary("MHA equivalent",        61, 2 * 128 * 128 * 2, 65_536))
print(kv_cache_summary("GQA equivalent (kv=8)", 61, 2 *   8 * 128 * 2, 65_536))
print(kv_cache_summary("MLA (latent=512+rope=64)", 61, (512 + 64) * 2, 65_536))

**Read the second block carefully.** DeepSeek-V3 with MLA fits ~4 GB of KV cache per 64k-context request. The equivalent MHA model would be 160 GB — fundamentally not deployable. GQA closes some of that gap; MLA closes most of it.

This is why every long-context frontier model in 2025–2026 has converged on aggressive KV compression.

## 4. FlashAttention vs naive at scale (GPU only)

Skip if you're on CPU. The interesting numbers come at seq lengths where the $O(n^2)$ memory penalty bites — 4k–16k tokens.

In [ ]:
if DEVICE != "cuda":
    print("CPU detected; skipping FA timing.")
    print("On A100 at seq=8192, expected speedup: ~4-7\u00d7 SDPA over naive.")
else:
    import time

    B, H, d = 2, 16, 64
    seq_lens = [1024, 2048, 4096, 8192]
    print(f"{'seq_len':>8s}  {'naive (ms)':>12s}  {'SDPA (ms)':>12s}  {'speedup':>8s}")

    for T in seq_lens:
        q = torch.randn(B, H, T, d, device=DEVICE, dtype=torch.bfloat16)
        k = torch.randn(B, H, T, d, device=DEVICE, dtype=torch.bfloat16)
        v = torch.randn(B, H, T, d, device=DEVICE, dtype=torch.bfloat16)

        # Warmup
        for _ in range(3):
            _ = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        torch.cuda.synchronize()

        # Naive
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(5):
            _ = scaled_dot_product_reference(q, k, v, is_causal=True)
        torch.cuda.synchronize(); naive_ms = (time.perf_counter() - t0) * 1000 / 5

        # SDPA
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(5):
            _ = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        torch.cuda.synchronize(); sdpa_ms = (time.perf_counter() - t0) * 1000 / 5

        print(f"{T:>8d}  {naive_ms:>12.2f}  {sdpa_ms:>12.2f}  {naive_ms/sdpa_ms:>7.2f}\u00d7")

On an A100 you should see the speedup grow with sequence length — the longer the sequence, the more the naive version pays for materializing the $n \times n$ attention matrix in HBM. By 16k tokens the naive version OOMs on most GPUs; SDPA happily runs at 128k.

## 5. MLA internals — what's actually cached

Inspect the shapes of the MLA forward pass to see exactly which tensors would live in the KV cache at inference.

In [ ]:
x = torch.randn(2, 16, 256)
freqs_cis_mla = precompute_freqs_cis(16, 16)
mla = MultiHeadLatentAttention(d_model=256, n_heads=8, d_head=32, d_rope=16, d_kv_latent=64)

# Manually replay the parts of forward() that produce cacheable state
with torch.no_grad():
    c_kv = mla.W_kv_down(x)                    # the latent (cached)
    k_r = mla.W_kr(x)                          # the shared RoPE key (cached)
    k_c_full = mla.W_kc(c_kv)                  # the *reconstructed* content K (NOT cached)
    v_full = mla.W_v(c_kv)                     # the *reconstructed* V         (NOT cached)

print("Cached (per layer, per token):")
print(f"  c_kv (latent):           {tuple(c_kv.shape[1:])}  = {c_kv.shape[-1]} values")
print(f"  k_r (shared RoPE key):   {tuple(k_r.shape[1:])}  = {k_r.shape[-1]} values")
print(f"  TOTAL CACHED PER TOKEN:                          {c_kv.shape[-1] + k_r.shape[-1]} values")
print()
print("Reconstructed on the fly at attention time (NOT cached):")
print(f"  k_c (content K, all heads): shape {tuple(k_c_full.shape[1:])}")
print(f"  v   (value,     all heads): shape {tuple(v_full.shape[1:])}")
print(f"  RECONSTRUCTION COST PER TOKEN:        {k_c_full.shape[-1] + v_full.shape[-1]} values")

The compression ratio is `(k_c + v) / (c_kv + k_r)`. In this toy config it's `(256 + 256) / (64 + 16) = 6.4×`. In DeepSeek-V3's real config it's ~57×.

The decompression (`W_kc @ c_kv`, `W_v @ c_kv`) happens inside the attention forward pass, so the *compute* cost is the same as full attention — only the cache is compressed. That's exactly the trade you want, since cache is what limits long-context inference.

## Recap

You now have:

- A working implementation of MHA, GQA, and MLA, all shape-verified.
- A FlashAttention pathway you can call without thinking (`F.scaled_dot_product_attention`).
- A mental model for the KV-cache cost across variants, in real architectural numbers.
- The conceptual landscape past MLA: hybrid attention, DSA, CSA.

**Next:** [Module 05 — Transformer Block](../05-transformer-block/). RMSNorm, SwiGLU, RoPE in depth, pre-norm vs post-norm — the rest of the block that wraps attention into a usable layer.